# 02. Walk-Forward Validation & Regime Verification

This notebook demonstrates our **Walk-Forward Validation Harness**. 
This harness simulates how the model would have performed historically in production by:
1. Establishing an initial training window (e.g. 3 years / 756 days).
2. Re-fitting both the scaler and HMM every quarter (63 trading days).
3. Generating out-of-sample regime predictions for the next quarter.
4. Concatenating the predictions and running an automated leakage check.
5. Comparing out-of-sample regimes side-by-side with the full-sample regimes.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ensure project root is in path
sys.path.append(os.path.abspath(".."))

from src.data_loader import load_and_align_data
from src.features import build_features, standardize_features
from src.regime_model import RegimeHMM
from src.validation import run_walk_forward_validation, run_leakage_check

## 1. Load Data & Build Features

In [ ]:
tickers = ["^NSEI", "GOLDBEES.NS", "LICNETFGSC.NS", "^INDIAVIX"]
start_date = "2010-01-01"
end_date = "2026-07-04"

# Load clean data (Dec 2019 Gold anomaly is auto-cleaned inside data_loader)
prices = load_and_align_data(tickers, start_date, end_date, cache_dir="../data/raw")
features = build_features(prices)
print(f"Dataset features prepared. Shape: {features.shape}")

## 2. Walk-Forward Validation

In [ ]:
initial_train_days = 756  # 3 years
refit_freq = 63           # 1 quarter

print("Running Walk-Forward Loop...")
oos_regimes, window_records = run_walk_forward_validation(
    features,
    initial_train_days=initial_train_days,
    refit_freq=refit_freq,
    random_state=100
)

oos_len = len(oos_regimes)
oos_start = oos_regimes.index.min().strftime('%Y-%m-%d')
oos_end = oos_regimes.index.max().strftime('%Y-%m-%d')

print(f"\nOut-of-sample backtest window: {oos_len} trading days ({oos_start} to {oos_end})")

## 3. Strict Leakage Verification

In [ ]:
# Run leakage check on 20 random dates to ensure training data always ends strictly before the date being predicted
run_leakage_check(oos_regimes, window_records, n_samples=20)

## 4. Full-Sample Fit (for comparison)

In [ ]:
# Standardize using full history (contains look-ahead/in-sample information)
fit_start = features.index.min().strftime("%Y-%m-%d")
fit_end = features.index.max().strftime("%Y-%m-%d")
std_features = standardize_features(features, fit_start, fit_end)

full_hmm = RegimeHMM(n_regimes=3, random_state=100)
full_hmm.fit(std_features)
full_regimes = full_hmm.predict_labeled_regimes(std_features)

full_regimes_oos = full_regimes.loc[oos_regimes.index]

print("Regime distribution comparison (Out-of-Sample period):")
compare_df = pd.DataFrame({
    "Full-Sample": full_regimes_oos.value_counts(),
    "Walk-Forward OOS": oos_regimes.value_counts()
})
display(compare_df)

## 5. Visualizing Regime Disagreement (Lag Check)

In [ ]:
nifty_oos = prices.loc[oos_regimes.index, "^NSEI"]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 12), sharex=True)
color_map = {"Bull": "green", "Bear": "orange", "Crisis": "red"}

# Panel 1: Full Sample
ax1.plot(nifty_oos.index, nifty_oos.values, color="black", linewidth=1.5, label="NIFTY 50")
curr_reg = full_regimes_oos.iloc[0]
start_dt = full_regimes_oos.index[0]

# Legend blocks
ax1.axvspan(start_dt, start_dt, color="green", alpha=0.15, label="Bull")
ax1.axvspan(start_dt, start_dt, color="orange", alpha=0.15, label="Bear")
ax1.axvspan(start_dt, start_dt, color="red", alpha=0.15, label="Crisis")

for date, regime in full_regimes_oos.items():
    if regime != curr_reg:
        ax1.axvspan(start_dt, date, color=color_map.get(curr_reg, "gray"), alpha=0.15)
        curr_reg = regime
        start_dt = date
ax1.axvspan(start_dt, full_regimes_oos.index[-1], color=color_map.get(curr_reg, "gray"), alpha=0.15)
ax1.set_title("Full-Sample HMM (In-Sample/Look-Ahead Shading)", fontsize=12, fontweight="bold")
ax1.set_yscale("log")
ax1.legend(loc="upper left")
ax1.grid(True, alpha=0.3)

# Panel 2: Walk Forward
ax2.plot(nifty_oos.index, nifty_oos.values, color="black", linewidth=1.5, label="NIFTY 50")
curr_reg_oos = oos_regimes.iloc[0]
start_dt_oos = oos_regimes.index[0]

# Legend blocks
ax2.axvspan(start_dt_oos, start_dt_oos, color="green", alpha=0.15, label="Bull")
ax2.axvspan(start_dt_oos, start_dt_oos, color="orange", alpha=0.15, label="Bear")
ax2.axvspan(start_dt_oos, start_dt_oos, color="red", alpha=0.15, label="Crisis")

for date, regime in oos_regimes.items():
    if regime != curr_reg_oos:
        ax2.axvspan(start_dt_oos, date, color=color_map.get(curr_reg_oos, "gray"), alpha=0.15)
        curr_reg_oos = regime
        start_dt_oos = date
ax2.axvspan(start_dt_oos, oos_regimes.index[-1], color=color_map.get(curr_reg_oos, "gray"), alpha=0.15)
ax2.set_title("Walk-Forward HMM (Out-of-Sample/No Leakage Shading)", fontsize=12, fontweight="bold")
ax2.set_yscale("log")
ax2.legend(loc="upper left")
ax2.grid(True, alpha=0.3)
ax2.set_xlabel("Date")

plt.suptitle("Full-Sample vs. Walk-Forward Out-of-Sample Regime Shading", fontsize=14, fontweight="bold", y=0.96)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()